# Nano-Jev: один дополнительный проход

Проверяем, улучшится ли уже дообученная Nano clean-first при продолжении с меньшей скоростью обучения. Веса стартуют из завершённого первого запуска; состояние AdamW там не сохранялось, поэтому оптимизатор новый.

Те же 22 656 строк, кандидаты и исходная синтетическая validation из 150 строк. Чистые и шумные строки перемешаны с начала (seed=20261007); чистый старт уже пройден. FP32, lr=1e-5, effective batch=16, максимум ещё один проход / 1 416 обновлений.

Validation NLL каждые 142 обновления, patience=3, min_delta=0.005; стартовые веса участвуют в выборе. Никаких новых данных, поиска гиперпараметров или выбора по третьему каталогу.

## Среда

Используем ту же версию Transformers, что в первом обучении Nano. GPU-время обучения не является замером задержки CPU.

In [ ]:
import os, sys, json, hashlib, subprocess
from pathlib import Path
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'transformers==5.6.2', 'safetensors==0.6.2'])
import torch
assert torch.cuda.is_available(), 'Нужен GPU Kaggle'
print({'torch':torch.__version__, 'gpu':torch.cuda.get_device_name(0)})


## Проверка пакета и обучение

Модель получает прежние пары question/option и query; ID ответа служит только меткой loss. Промахи E5 на валидации остаются ошибками цепочки, но не дают ложную метку выбора.

In [ ]:
matches = list(Path('/kaggle/input').rglob('nano-continuation-input-manifest.json'))
assert len(matches)==1, f'Ожидался один пакет данных: {len(matches)}'
input_dir = matches[0].parent
assert hashlib.sha256(matches[0].read_bytes()).hexdigest()=='0cea12c910fa27789e3f7fe2729aee9f76e9000faa52c65069bc467efffa0759', 'Другой пакет данных'
output_dir = Path('/kaggle/working/results')
command = [sys.executable, str(input_dir/'continue_nanojev_kaggle.py'), '--input', str(input_dir), '--output', str(output_dir)]
with Path('/kaggle/working/training.log').open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    assert process.wait()==0, 'Ошибка обучения; см. training.log'


## Выбранные и последние веса

Сохраняем обе версии и полный журнал. Если best_step=0, стартовые веса оказались лучше по выбранному критерию. Остальные известные наборы сравним локально после фиксации checkpoint; это повторная проверка качества, а не новый слепой тест.

In [ ]:
result = json.loads((output_dir/'result.json').read_text())
print({k:result[k] for k in ('steps','best_step','stop_reason','elapsed_seconds')})
print({'baseline_nll':result['baseline']['selection_loss'], 'best_nll':result['best']['selection_loss']})
for profile in result['baseline']['metrics']:
    before = result['baseline']['metrics'][profile]['accuracy']
    after = result['best']['metrics'][profile]['accuracy']
    print(f'{profile}: {before:.1%} → {after:.1%}')
